# Future Crop Challenge — per-location mechanistic yield model

Per-cell yield model fit on **all 39 training years (381–419)** and applied to
the full test window (420–497). Each grid cell gets its own parameter vector,
fit independently — no pooling across locations. Two crop-specific models are
used, chosen on the sandbox benchmark (median per-cell R² on an 8-year
held-out; see `sandbox/ANALYSIS.md`):

| crop | model | form |
|---|---|---|
| wheat | `13_water_heat_bilinear` | `a + b·tanh(GDD/1000−1) + c·tanh(GDD/1000−1)·tanh(PREC/10) + d·log1p(PREC_MID) − e·tanh(VD/50)` |
| maize | `06_saturating_vpd` | `a + b·tanh(GDD/1000−1) + c·log1p(PREC) − d·HEAT30/8 − e·tanh(VPD/2.5) + f·log(CO2/380)` |

**Why this shape.** Per-cell held-out R² decomposes into a *level-anchor* term
(the drift between the training-window mean and the scoring window) plus a
*weather* term. The weather term is real on maize (VPD/heat/water) but ~absent
on wheat. CO₂ fertilization appears in the data but only maize shows any
response to it, so only `06` carries a `log(CO2/380)` term; wheat's candidates
all fit it to ~zero. The sun terms (`rsds`) and continuous VPD were screened
out — they regress vs. the VD/50 exceedance index.

**Fit protocol.** Params enter the models linearly once the weather features
are computed, so the sandbox objective
`min mean((pred−y)²) + ridge·Σ(params[1:]²)` (ridge = 0.1, intercept exempt,
fit by L-BFGS-B in the sandbox) has a closed form
`θ = (XᵀX + T·ridge·R)⁻¹ Xᵀy`, `R = diag(0,1,…,1)` — verified numerically to
<1e-5 against L-BFGS-B. We solve that directly (fast, exact, no optimizer).
Prediction is floored at 0.


In [ ]:
import numpy as np
import pandas as pd
import time

t0 = time.time()

# ---- point DATA_DIR at the local repo for testing ----
DATA_DIR = "/kaggle/input/the-future-crop-challenge/"     # Kaggle
# DATA_DIR = "data/"                                      # local run

RIDGE = 0.1              # ridge weight on params[1:]^2 (intercept exempt)
MIN_TRAIN_YEARS = 3      # per-cell min train rows to fit; fewer -> cell mean


## Data loading

Each feature is one parquet per (variable, crop, split). Row layout is uniform
across files and aligned by the row **index = submission ID**: columns `'0'…'239'`
are the 240-day series. The `soil_co2_*.parquet` files carry the per-row
metadata (`year`, `lon`, `lat`, atmospheric `co2`); `pr` is stored in m/day
and multiplied by 1000 → mm/day to match the sandbox cache pipeline.

Running locally: uncomment the `data/` `DATA_DIR` line. On Kaggle the default
`/kaggle/input/the-future-crop-challenge/` is used.


In [ ]:
def read_meta(crop, split):
    """Metadata carrier parquet: year, lon, lat, atmospheric CO2 per row."""
    df = pd.read_parquet(f"{DATA_DIR}/soil_co2_{crop}_{split}.parquet")
    return df[["year", "lon", "lat", "co2"]]


def read_days(feature, crop, split):
    """Day columns '0'..'239' of one climate feature as float32 (N, 240)."""
    df = pd.read_parquet(f"{DATA_DIR}/{feature}_{crop}_{split}.parquet",
                         columns=[str(i) for i in range(240)])
    return df.to_numpy(dtype=np.float32)


def location_codes(meta_tr, meta_te):
    """Consistent per-cell codes across train and test rows (test subset of train)."""
    both = pd.concat([meta_tr[["lon", "lat"]], meta_te[["lon", "lat"]]],
                     ignore_index=True)
    codes = both.groupby(["lon", "lat"], sort=False).ngroup().to_numpy()
    return codes[: len(meta_tr)], codes[len(meta_tr):]


def make_design(crop, tasmax, tasmin, pr_mm, co2):
    """Model design matrix; parameters enter linearly -> closed-form ridge.

    wheat -> 13_water_heat_bilinear (no CO2: wheat ignores it in the sandbox):
        a + b*tanh(GDD/1000-1) + c*tanh(GDD/1000-1)*tanh(PREC/10)
          + d*log1p(PRC_MID) - e*tanh(VD/50)
    maize -> 06_saturating_vpd (the only candidate with a CO2 term):
        a + b*tanh(GDD/1000-1) + c*log1p(PREC) - d*HEAT30/8
          - e*tanh(VPD/2.5) + f*log(CO2/380)
    """
    tmean = 0.5 * (tasmax + tasmin)
    gdd = np.maximum(tmean - 8.0, 0.0).sum(axis=1)
    prec = pr_mm.sum(axis=1)
    prc_mid = pr_mm[:, 80:160].sum(axis=1)
    es = lambda T: 0.6108 * np.exp(17.27 * T / (T + 237.3))   # Magnus, hPa
    vpd = es(tasmax) - es(tasmin)
    heat30 = (tasmax > 30.0).sum(axis=1)
    vpd_mean = vpd.mean(axis=1)
    vd = np.maximum(vpd - 2.0, 0.0).sum(axis=1)
    th = np.tanh(gdd / 1000.0 - 1.0)
    if crop == "wheat":
        return np.column_stack([np.ones_like(th), th,
                                th * np.tanh(prec / 10.0),
                                np.log1p(prc_mid), -np.tanh(vd / 50.0)])
    return np.column_stack([np.ones_like(th), th, np.log1p(prec),
                            -(heat30 / 8.0), -np.tanh(vpd_mean / 2.5),
                            np.log(np.maximum(co2, 1e-6) / 380.0)])


def fit_cells(D_tr, y_tr, loc, ridge=RIDGE):
    """Per-cell closed-form ridge: min mean||X@th - y||^2 + ridge*sum(th[1:]^2).

    Exact optimum of the sandbox L-BFGS-B objective (models are linear in the
    parameters once the weather features are fixed).
    """
    n_locs = loc.max() + 1
    p = D_tr.shape[1]
    R = np.diag([0.0] + [1.0] * (p - 1))
    params = np.zeros((n_locs, p))
    for c in range(n_locs):
        m = np.where(loc == c)[0]
        yc = y_tr[m]
        if len(m) < MIN_TRAIN_YEARS:
            params[c, 0] = float(yc.mean()) if len(m) else np.nan
            continue
        Xc = D_tr[m]
        try:
            params[c] = np.linalg.solve(Xc.T @ Xc + len(m) * ridge * R,
                                        Xc.T @ yc)
        except np.linalg.LinAlgError:
            params[c, 0] = float(yc.mean())
    return params


def predict_split(crop, tasmax, tasmin, pr_mm, co2, params, loc):
    D = make_design(crop, tasmax, tasmin, pr_mm, co2)
    pred = (D * params[loc]).sum(axis=1)
    return np.maximum(pred, 0.0)


## Fit and predict

For each crop we (1) build a per-row design matrix from the weather features,
(2) fit one ridge-parameter vector per grid cell on train years 381–419,
(3) predict every test row (420–497) with its cell's params, floored at 0.


In [ ]:
preds, crop_stats = {}, {}
for crop in ["maize", "wheat"]:
    print(f"== {crop} ==", flush=True)

    meta_tr = read_meta(crop, "train")
    meta_te = read_meta(crop, "test")
    loc_tr, loc_te = location_codes(meta_tr, meta_te)

    tasmax = read_days("tasmax", crop, "train")
    tasmin = read_days("tasmin", crop, "train")
    pr = read_days("pr", crop, "train") * 1000.0              # m/day -> mm
    y_tr = pd.read_parquet(f"{DATA_DIR}/train_solutions_{crop}.parquet")["yield"].to_numpy()
    D_tr = make_design(crop, tasmax, tasmin, pr, meta_tr["co2"].to_numpy())
    params = fit_cells(D_tr, y_tr, loc_tr)
    del tasmax, tasmin, pr, D_tr

    tasmax = read_days("tasmax", crop, "test")
    tasmin = read_days("tasmin", crop, "test")
    pr = read_days("pr", crop, "test") * 1000.0
    pred = predict_split(crop, tasmax, tasmin, pr,
                         meta_te["co2"].to_numpy(), params, loc_te)
    del tasmax, tasmin, pr

    seen = set(np.unique(loc_tr).tolist())
    unseen = np.array([c not in seen for c in loc_te])
    if unseen.any():
        crop_mean = float(np.mean(y_tr))
        pred[unseen] = crop_mean
        print(f"  WARNING: {unseen.sum()} test rows in cells with no train data", flush=True)

    preds[crop] = pd.Series(pred, index=meta_te.index, name="yield")
    crop_stats[crop] = dict(n_pred=len(pred),
                            pred_mean=float(pred.mean()),
                            pred_std=float(pred.std()),
                            train_mean=float(np.mean(y_tr)),
                            n_floored=int((pred == 0.0).sum()))
    print(f"  rows={len(pred)}  pred_mean={pred.mean():.3f} "
          f"train_mean={np.mean(y_tr):.3f}  floored_at_0={int((pred == 0).sum())}",
          flush=True)


## Assemble submission

Merge predictions (keyed by test ID = parquet index) onto the sample
submission so every row gets a yield. Defensive gap-fill (should not trigger)
uses the overall train mean.


In [ ]:
sub = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
pred_all = pd.concat([preds["maize"], preds["wheat"]])
sub["yield"] = sub["ID"].map(pred_all)

missing = int(sub["yield"].isna().sum())
if missing:
    sub["yield"] = sub["yield"].fillna(sub["yield"].mean())
    print(f"WARNING: {missing} rows had no prediction, filled with mean", flush=True)

sub.to_csv("submission.csv", index=False)

print("\n= submission =")
print(f"rows: {len(sub)}  NaNs: {missing}  elapsed: {time.time() - t0:.0f}s")
for crop, s in crop_stats.items():
    print(f"{crop:6s} n={s['n_pred']:7d}  pred_mean={s['pred_mean']:.3f}  "
          f"pred_std={s['pred_std']:.3f}  train_mean={s['train_mean']:.3f}  "
          f"floored_at_0={s['n_floored']}")
print(sub.head())


## Notes / caveats

- **Extrapolation**: the test window (420–497) is ~80 years; CO₂ roughly
  doubles (418 → 1108 ppm), far beyond the train range (341–415). The maize
  model carries a `log(CO2/380)` term, but its fitted coefficient is ~0 — the
  sandbox repeatedly found CO₂ terms dead on both crops — so the doubling does
  not rescue the forecast.
- **What the models actually forecast** (mean of row predictions):
  - **maize**: −19% (train mean 3.70 → test mean 2.98). The entire decline is
    the heat term: 30 °C+ hot-day count scales as HEAT30/8 mean 10.3 → 15.0
    (+46%), and with the fitted coefficient (≈0.15) that alone is −0.70 yield.
    GDD, mean VPD and precipitation contributions are all near zero in the net.
  - **wheat**: flat (2.81 → 2.82). Weather terms are fitted to ~zero (the
    benchmark's "no wheat weather skill"), so predictions ride the per-cell
    intercept.
- **Level anchor**: the per-cell intercept absorbs each location's long-run
  yield. Because the scoring window is ~80 years, drift from the training mean
  matters far less than in the 8-year sandbox split — the level term is mostly
  in-sample, and the weather term is what separates models.
- **Deterministic & fast**: closed-form ridge per cell — no optimizer — a
  couple of minutes on Kaggle CPU.
- Reproduce locally from the repo checkout by setting `DATA_DIR = "data/"`.
